# Module 03: Matplotlib for Machine Learning
## Notebook 02: Core Statistical Plots for Exploratory Data Analysis

Before applying any machine learning algorithm, you must visually explore your feature distributions, check for skewness, spot anomalous outliers, and examine relationships between variables.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Construct multi-dimensional **Scatter Plots** using color mappings and marker sizes.
2. Build horizontal and vertical **Bar Charts** for categorical feature analysis and feature importances.
3. Plot **Histograms with Density Estimations** to inspect probability distributions.
4. Construct **Box-and-Whisker Plots** to detect quartile skewness and outliers.
5. **Advanced:** Render **Hexagonal 2D Binning (`ax.hexbin`)** with logarithmic normalization to eliminate scatter plot overplotting on massive ($N=100,000$) datasets.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print(f"Matplotlib loaded!")

### 1. Scatter Plots: Visualizing Feature Relationships

In classification and clustering:
- $x$-axis and $y$-axis show two continuous features.
- Color (`c=...`) maps to a target class or continuous 3rd variable.
- Colormap (`cmap='viridis'` or `'coolwarm'`) enhances perceptual contrast.
- Point size (`s=...`) can represent a 4th feature dimension!

In [ ]:
rng = np.random.default_rng(42)
N = 150

# Features: House Size, Bedrooms, Price (Target)
house_size = rng.uniform(800, 3500, size=N)
bedrooms = rng.integers(1, 6, size=N)
price = house_size * 180 + bedrooms * 25000 + rng.normal(0, 30000, size=N)

fig, ax = plt.subplots(figsize=(8.5, 5))

# Scatter with colormap mapped to bedrooms
scatter = ax.scatter(
    house_size, 
    price, 
    c=bedrooms, 
    cmap='plasma', 
    s=bedrooms * 25, 
    alpha=0.8, 
    edgecolors='k',
    linewidth=0.5
)

# Add colorbar
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Number of Bedrooms", fontsize=10)

ax.set_title("Housing Market: Size vs. Price by Bedroom Count", fontsize=13, fontweight='bold')
ax.set_xlabel("Living Area (Square Feet)", fontsize=11)
ax.set_ylabel("Selling Price ($)", fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5)

plt.show()

---
### 2. Bar Charts: Feature Importances and Categorical Frequencies

In ML, bar charts are most frequently used to display:
- Class distribution / target balance.
- Model feature importances (e.g., from Random Forests or Lasso).
- Horizontal bars (`ax.barh`) are ideal when feature names are long.

In [ ]:
features = ['Median_Income', 'House_Age', 'Ave_Rooms', 'Ave_Bedrms', 'Population', 'Latitude', 'Longitude']
importances = np.array([0.38, 0.12, 0.08, 0.04, 0.06, 0.17, 0.15])

# Sort features by importance ascending for clean horizontal presentation
sorted_idx = np.argsort(importances)
sorted_features = [features[i] for i in sorted_idx]
sorted_importances = importances[sorted_idx]

fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.barh(sorted_features, sorted_importances, color='steelblue', edgecolor='black', height=0.6)
ax.set_title("Random Forest Feature Importances", fontsize=13, fontweight='bold')
ax.set_xlabel("Relative Importance Score", fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.6)

# Annotate values at the end of each bar
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.008, bar.get_y() + bar.get_height()/2, f"{width:.2f}", va='center', fontsize=9)

ax.set_xlim(0, 0.45)
plt.show()

---
### 3. Histograms and Box Plots: Inspecting Distributions and Outliers

- **Histograms (`ax.hist`)**: Reveal skewness, modality (unimodal vs bimodal), and tail heaviness.
- **Box Plots (`ax.boxplot`)**: Compactly visualize the median, Interquartile Range (IQR box), and individual outlier points beyond $1.5 \times IQR$.

In [ ]:
# Generate normal and skewed distributions
normal_feature = rng.normal(loc=50, scale=10, size=500)
skewed_feature = rng.exponential(scale=20, size=500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# 1. Histogram
ax1.hist(normal_feature, bins=25, color='cornflowerblue', edgecolor='black', alpha=0.7, density=True, label='Gaussian')
ax1.set_title("Feature Distribution (Histogram)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Value")
ax1.set_ylabel("Probability Density")
ax1.grid(True, linestyle=':', alpha=0.5)

# 2. Box Plot comparing both features
ax2.boxplot([normal_feature, skewed_feature], tick_labels=['Normal Feature', 'Skewed Feature'], patch_artist=True,
            boxprops=dict(facecolor='lightgray', color='navy'),
            medianprops=dict(color='crimson', linewidth=2))
ax2.set_title("Outlier & Spread Audit (Box Plot)", fontsize=12, fontweight='bold')
ax2.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

---
### 4. Advanced Complex Usage: Hexagonal 2D Density Estimation for Massive Datasets

When datasets scale to $N = 100,000+$ samples:
- Traditional scatter plots suffer catastrophic **overplotting**: points overlap into an opaque, solid blob where relative density is completely lost.
- **`ax.hexbin()`**: Tessellates the 2D plane into regular hexagonal bins, counting the number of observations within each cell.
- Combined with **Logarithmic Color Normalization (`matplotlib.colors.LogNorm`)**, it clearly reveals low-density outlier tails and high-density cores spanning multiple orders of magnitude.

In [ ]:
import matplotlib.colors as mcolors

# Simulate 100,000 samples with a dense core and non-linear correlation
N_large = 100_000
x_mass = rng.normal(loc=0.0, scale=1.0, size=N_large)
y_mass = x_mass**2 + rng.normal(loc=0.0, scale=0.5, size=N_large)

fig, (ax_scatter, ax_hex) = plt.subplots(1, 2, figsize=(13, 5))

# 1. Failure case: Scatter plot with severe overplotting
ax_scatter.scatter(x_mass[:10000], y_mass[:10000], alpha=0.2, s=2, color='navy')
ax_scatter.set_title("Failure Case: Scatter Plot (Overplotting)", fontsize=12, fontweight='bold')
ax_scatter.set_xlabel("Feature X")
ax_scatter.set_ylabel("Feature Y")
ax_scatter.grid(True, linestyle=':', alpha=0.4)

# 2. Production solution: Hexbin with Logarithmic Normalization
hb = ax_hex.hexbin(
    x_mass, 
    y_mass, 
    gridsize=45, 
    cmap='viridis', 
    norm=mcolors.LogNorm(),  # Log-scale colorbar for orders of magnitude
    mincnt=1
)
cbar = fig.colorbar(hb, ax=ax_hex)
cbar.set_label("Sample Count per Hex Bin (Log Scale)", fontsize=10)

ax_hex.set_title("Hexbin 2D Density Map (N=100,000 samples)", fontsize=12, fontweight='bold')
ax_hex.set_xlabel("Feature X")
ax_hex.set_ylabel("Feature Y")
ax_hex.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

### Summary & Next Steps
In this notebook, you mastered:
- Multi-dimensional scatter plots with colormaps and marker scaling.
- Horizontal bar charts with direct numerical value annotations.
- Histograms with density scaling and box plots for visual IQR audits.
- Hexagonal 2D density estimation (`ax.hexbin` with `LogNorm`) for high-volume datasets.

**Next Notebook:** `03_multi_plot_layouts_and_subplots.ipynb` — Build multi-panel figure dashboards, asymmetric layouts using `GridSpec`, and inset zoom axes.